In [1]:
import os
from openai import OpenAI
from dotenv import load_dotenv

from google.adk.agents import Agent
from google.adk.models.lite_llm import LiteLlm

from google.adk.sessions import InMemorySessionService
from google.adk.runners import Runner
from google.adk.agents.run_config import RunConfig
from google.genai import types

from utils.tools import check_warehouse_availability, reserve_warehouse_items

d:\ai_projects\agentic_rag\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv()
openai_client = OpenAI()

### ADK Agent

In [4]:
model = LiteLlm(
    model="openai/gpt-4.1",
    temperature=0.0,
    # api_Key=os.getenv("OPENAI_API_KEY"),
)

In [5]:
warehouse_agent = Agent(
    name="warehouse_manager_agent",
    model=model,
    tools=[check_warehouse_availability, reserve_warehouse_items],
    description="A agent that can check the availability of items in the warehouses and reserve them.",
    instruction="""
You are a part of the shopping assistant that can manage available inventory in the warehouses.

You will be given a conversation history and a list of tools, your task is to perform the action requested by the latest user query. Answer part of the query that you can answer with the available tools.

Instructions:
- You must always check the availability of the items in the warehouses before reserving them.
- Only reserve items in warehouses if entire order can be reserved or the user has confirmed that they want a partial reservation.
- If you cannot reserve any items, return an answer that the order cannot be reserved.
- If you can reserve some items, return an answer that the order can be partially reserved and include the details.
- If only partial quantity can be reserved in some warehouses, try to combine the required quantity from different warehouses.
- Try to reserve items from the closest warehouse to the user first if users location is provided.
"""
)

### ADK Session

In [6]:
session_service = InMemorySessionService()

In [7]:
await session_service.create_session(
    app_name="warehouse_app",
    user_id="user_1",
    session_id="session_1"
)

Session(id='session_1', app_name='warehouse_app', user_id='user_1', state={}, events=[], last_update_time=1784077356.2187362)

In [8]:
runner = Runner(
    agent=warehouse_agent,
    session_service=session_service,
    app_name="warehouse_app",
)

In [9]:
message = types.Content(
    role="user",
    parts=[types.Part(text="what is the availability of B09WCL37Z4 in all of the warehouses?")]
)

In [10]:
result = runner.run(
    user_id="user_1",
    session_id="session_1",
    new_message=message,
    run_config=RunConfig(
        max_llm_calls=3,
    )
)

In [11]:
for event in result:
    print("================================================")
    print(event)

d:\ai_projects\agentic_rag\.venv\Lib\site-packages\google\adk\tools\function_tool.py:95: UserWarning: [EXPERIMENTAL] feature FeatureName.JSON_SCHEMA_FOR_FUNC_DECL is enabled.
  build_function_declaration(


model_version='gpt-4.1-2025-04-14' content=Content(
  parts=[
    Part(
      function_call=FunctionCall(
        args={
          'items': [
            {<... 2 items at Max depth ...>},
          ]
        },
        id='call_VUscAiLo3w3cCdQjPtEoj5KT',
        name='check_warehouse_availability'
      )
    ),
  ],
  role='model'
) grounding_metadata=None partial=False turn_complete=None turn_complete_reason=None finish_reason=<FinishReason.STOP: 'STOP'> error_code=None error_message=None interrupted=None custom_metadata=None usage_metadata=GenerateContentResponseUsageMetadata(
  cached_content_token_count=0,
  candidates_token_count=32,
  prompt_token_count=568,
  total_token_count=600
) live_session_resumption_update=None live_session_id=None go_away=None input_transcription=None output_transcription=None avg_logprobs=None logprobs_result=None cache_metadata=None citation_metadata=None interaction_id=None invocation_id='e-b2523520-0bc2-454e-a62c-c1257991cee9' author='warehouse_mana

In [13]:
session_service = InMemorySessionService()

async def ask_warehouse_agent(query: str, session_id: str):

    # Check if session exists, only create if it doesn't
    existing_session = await session_service.get_session(
        app_name="warehouse_app",
        user_id="user_1",
        session_id=session_id,
    )

    if not existing_session:
        await session_service.create_session(
            app_name="warehouse_app",
            user_id="user_1",
            session_id=session_id,
        )

    runner= Runner(
        agent=warehouse_agent,
        app_name="warehouse_app",
        session_service=session_service,
    )

    content = types.Content(
        role="user",
        parts=[types.Part(text=query)],
    )

    final_text = ""

    for event in runner.run(
        user_id="user_1",
        session_id=session_id,
        new_message=content,
        run_config=RunConfig(
            max_llm_calls=3
        )
    ):
        if event.is_final_response():
            if event.content and event.content.parts:
                for part in event.content.parts:
                    final_text += part.text
            break
    
    return final_text

In [14]:
answer = await ask_warehouse_agent("What is the availability of B09WCL37Z4 in all of the warehouses?", session_id ="123")

In [15]:
print(answer)

The product B09WCL37Z4 is available in the following warehouses:

- Berlin Distribution Center (Berlin, Germany): 8 units available
- Guangzhou North Warehouse (Guangzhou, China): 57 units available
- Munich Logistics Hub (Munich, Germany): 72 units available
- Fujian Central Depot (Fujian, China): 50 units available

It is not available in the Lyon Regional Warehouse or the Marseille Mediterranean Hub in France. If you need a specific quantity or want to reserve from a particular warehouse, please let me know!


In [16]:
answer = await ask_warehouse_agent("Can you reserve 2 in Fujian?", session_id ="123")

In [17]:
print(answer)

2 units of product B09WCL37Z4 have been successfully reserved for you at the Fujian Central Depot (Fujian, China). If you need further assistance or wish to reserve more, please let me know!


#### uv run adk web --port 8010 to activate ADK development platform at 127.0.0.1:8010